## 1. Imports


In [ ]:
import os
from pathlib import Path
import time
import warnings
import logging

import joblib
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import numpy as np
import pandas as pd
import plotly.express as px
import shap
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
MLFLOW_DIR = (PROJECT_ROOT / "mlruns").resolve()
MLFLOW_URI = MLFLOW_DIR.as_uri()
DATA_FILES = [
    DATA_DIR / "raw" / "HI-Small_Trans.csv",
    DATA_DIR / "raw" / "LI-Small_Trans.csv",
]



EXPERIMENT = "aml-detection"
RANDOM_STATE = 42
N_SPLITS = 5

MODEL_DIR.mkdir(parents=True, exist_ok=True)

COLS = [
    "Timestamp",
    "From Bank",
    "From Account",
    "To Bank",
    "To Account",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]


## 2. Data Loading


In [ ]:
def load_data(files: list[str]) -> pd.DataFrame:
    frames = []

    for path in files:
        if not os.path.exists(path):
            log.warning(f"File not found, skip: {path}")
            continue

        log.info(f"Reading {path} ...")
        started_at = time.time()
        df = pd.read_csv(
            path,
            names=COLS,
            header=0,
            dtype={
                "From Bank": str,
                "From Account": str,
                "To Bank": str,
                "To Account": str,
                "Payment Format": str,
                "Receiving Currency": str,
                "Payment Currency": str,
                "Is Laundering": np.int8,
            },
            low_memory=False,
        )

        elapsed = time.time() - started_at
        log.info(f"  -> {len(df):,} rows [{elapsed:.1f}s]")
        frames.append(df)

    if not frames:
        raise ValueError("No input files were loaded. Check DATA_FILES paths.")

    data = pd.concat(frames, ignore_index=True)
    data["Timestamp"] = pd.to_datetime(data["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce")
    data["Amount Received"] = pd.to_numeric(data["Amount Received"], errors="coerce")
    data["Amount Paid"] = pd.to_numeric(data["Amount Paid"], errors="coerce")

    laundering_rate = data["Is Laundering"].mean() * 100
    log.info(f"Total rows: {len(data):,} | Laundering rate: {laundering_rate:.3f}%")
    return data


## 3. Exploratory Data Analysis


In [ ]:
df_raw = load_data(DATA_FILES)
df_raw.head()


In [ ]:
overview = {
    "rows": len(df_raw),
    "columns": len(df_raw.columns),
    "laundering_rate": float(df_raw["Is Laundering"].mean()),
    "start_ts": df_raw["Timestamp"].min(),
    "end_ts": df_raw["Timestamp"].max(),
}
overview


In [ ]:
pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "missing": df_raw.isna().sum(),
        "unique_values": df_raw.nunique(dropna=False),
    }
)


In [ ]:
class_df = pd.DataFrame(
    {
        "class": ["Normal", "Laundering"],
        "count": [
            int((df_raw["Is Laundering"] == 0).sum()),
            int((df_raw["Is Laundering"] == 1).sum()),
        ],
    }
)

fig = px.bar(class_df, x="class", y="count", color="class", text="count", title="Class Balance")
fig.update_yaxes(type="log")
fig.show()


### Что показывает график `Class Balance`

- Этот график сравнивает количество обычных транзакций (`Normal`) и подозрительных (`Laundering`).
- Ось `Y` здесь логарифмическая, поэтому разница между классами визуально сжата. Это сделано специально: без логарифмической шкалы столбец подозрительных операций был бы почти не виден.
- Главный вывод: в данных очень сильный дисбаланс классов. Подозрительных операций крайне мало, поэтому модель нельзя оценивать только по `accuracy` - важнее смотреть на `recall`, `precision`, `PR-AUC` и порог классификации.


In [ ]:
daily_stats = (
    df_raw.assign(date=df_raw["Timestamp"].dt.floor("D"))
    .groupby("date")
    .agg(
        transactions=("Is Laundering", "size"),
        laundering_rate=("Is Laundering", "mean"),
    )
    .reset_index()
)

fig = px.line(daily_stats, x="date", y="laundering_rate", title="Daily Laundering Rate")
fig.show()


### Что показывает график `Daily Laundering Rate`

- Здесь по оси `X` идут дни, а по оси `Y` - доля подозрительных транзакций в этот день.
- График нужен, чтобы понять, равномерно ли распределены подозрительные операции по времени или есть отдельные дни со всплесками.
- Если на линии видны пики, это может означать, что в некоторые дни поведение потока заметно меняется. Для модели это сигнал, что временные признаки (`day`, `hour`, `weekday`) действительно могут быть полезны.


In [ ]:
hourly_stats = (
    df_raw.groupby(df_raw["Timestamp"].dt.hour.rename("hour"))
    .agg(
        transactions=("Is Laundering", "size"),
        laundering_rate=("Is Laundering", "mean"),
    )
    .reset_index()
)

fig = px.line(hourly_stats, x="hour", y="laundering_rate", markers=True, title="Hourly Laundering Rate")
fig.show()


### Что показывает график `Hourly Laundering Rate`

- Этот график показывает, как меняется доля подозрительных операций в зависимости от часа суток.
- Он помогает увидеть, есть ли у подозрительных транзакций характерные временные окна, например ночные часы или периоды низкой обычной активности.
- Если в определённые часы доля `laundering` выше, это подтверждает полезность признаков `hour` и `is_night` в модели.


In [ ]:
amount_plot = df_raw[["Amount Paid", "Is Laundering"]].dropna().copy()
amount_plot["log_amount_paid"] = np.log1p(amount_plot["Amount Paid"].clip(lower=0))
amount_plot["label"] = amount_plot["Is Laundering"].map({0: "Normal", 1: "Laundering"})

fig = px.histogram(
    amount_plot,
    x="log_amount_paid",
    color="label",
    barmode="overlay",
    histnorm="probability density",
    nbins=60,
    title="Distribution of log(Amount Paid + 1)",
)
fig.show()


### Что показывает график `Distribution of log(Amount Paid + 1)`

- На этом графике сравнивается распределение сумм платежей для обычных и подозрительных транзакций.
- Используется преобразование `log(Amount Paid + 1)`, чтобы очень большие суммы не "раздавили" весь график и можно было сравнивать формы распределений.
- Если кривые для `Normal` и `Laundering` заметно различаются, значит размер платежа может быть полезным сигналом для модели. Если они сильно перекрываются, одной только суммы недостаточно и нужно опираться на комбинацию признаков.


In [ ]:
payment_format_stats = (
    df_raw.groupby("Payment Format")
    .agg(
        rows=("Is Laundering", "size"),
        laundering_rate=("Is Laundering", "mean"),
    )
    .sort_values(["laundering_rate", "rows"], ascending=[False, False])
    .reset_index()
)
payment_format_stats.head(15)


In [ ]:
currency_stats = (
    df_raw.groupby("Payment Currency")
    .agg(
        rows=("Is Laundering", "size"),
        laundering_rate=("Is Laundering", "mean"),
    )
    .sort_values(["laundering_rate", "rows"], ascending=[False, False])
    .reset_index()
)
currency_stats.head(15)


## 4. Feature Engineering


In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    log.info("Engineering features...")
    data = df.copy()

    data["hour"] = data["Timestamp"].dt.hour.astype(np.int8)
    data["day_of_week"] = data["Timestamp"].dt.dayofweek.astype(np.int8)
    data["day_of_month"] = data["Timestamp"].dt.day.astype(np.int8)
    data["is_weekend"] = (data["day_of_week"] >= 5).astype(np.int8)
    data["is_night"] = ((data["hour"] < 6) | (data["hour"] >= 22)).astype(np.int8)

    data["Amount Received"] = pd.to_numeric(data["Amount Received"], errors="coerce").fillna(0)
    data["Amount Paid"] = pd.to_numeric(data["Amount Paid"], errors="coerce").fillna(0)
    data["amount_ratio"] = np.where(
        data["Amount Paid"] > 0,
        data["Amount Received"] / (data["Amount Paid"] + 1e-9),
        1.0,
    )
    data["amount_diff"] = data["Amount Received"] - data["Amount Paid"]
    data["log_amount_paid"] = np.log1p(data["Amount Paid"])
    data["log_amount_recv"] = np.log1p(data["Amount Received"])
    data["is_round_amount"] = (data["Amount Paid"] % 1000 == 0).astype(np.int8)

    data["currency_mismatch"] = (data["Receiving Currency"] != data["Payment Currency"]).astype(np.int8)
    data["same_bank"] = (data["From Bank"] == data["To Bank"]).astype(np.int8)
    data["self_transfer"] = (data["From Account"] == data["To Account"]).astype(np.int8)

    log.info("  Computing sender aggregates...")
    sender_stats = (
        data.groupby("From Account")
        .agg(
            sender_tx_count=("Amount Paid", "count"),
            sender_total_paid=("Amount Paid", "sum"),
            sender_mean_paid=("Amount Paid", "mean"),
            sender_std_paid=("Amount Paid", "std"),
            sender_unique_recv=("To Account", "nunique"),
        )
        .reset_index()
        .rename(columns={"From Account": "account_key"})
    )

    log.info("  Computing receiver aggregates...")
    receiver_stats = (
        data.groupby("To Account")
        .agg(
            recv_tx_count=("Amount Received", "count"),
            recv_total_received=("Amount Received", "sum"),
            recv_mean_received=("Amount Received", "mean"),
            recv_unique_sender=("From Account", "nunique"),
        )
        .reset_index()
        .rename(columns={"To Account": "account_key"})
    )

    data = data.merge(sender_stats, left_on="From Account", right_on="account_key", how="left").drop(columns="account_key")
    data = data.merge(receiver_stats, left_on="To Account", right_on="account_key", how="left").drop(columns="account_key")

    data["fanout_ratio"] = data["sender_unique_recv"] / (data["sender_tx_count"] + 1)
    data["fanin_ratio"] = data["recv_unique_sender"] / (data["recv_tx_count"] + 1)
    data["sender_std_paid"] = data["sender_std_paid"].fillna(0)

    categorical_cols = [
        "Payment Format",
        "Receiving Currency",
        "Payment Currency",
        "From Bank",
        "To Bank",
    ]
    for col in categorical_cols:
        encoder = LabelEncoder()
        data[f"{col}_enc"] = encoder.fit_transform(data[col].fillna("UNK").astype(str))

    log.info(f"  Features ready. Shape: {data.shape}")
    return data


## 5. Feature List and Threshold Search


In [ ]:
FEATURE_COLS = [
    "hour",
    "day_of_week",
    "day_of_month",
    "is_weekend",
    "is_night",
    "amount_ratio",
    "amount_diff",
    "log_amount_paid",
    "log_amount_recv",
    "is_round_amount",
    "currency_mismatch",
    "same_bank",
    "self_transfer",
    "sender_tx_count",
    "sender_total_paid",
    "sender_mean_paid",
    "sender_std_paid",
    "sender_unique_recv",
    "fanout_ratio",
    "recv_tx_count",
    "recv_total_received",
    "recv_mean_received",
    "recv_unique_sender",
    "fanin_ratio",
    "Payment Format_enc",
    "Receiving Currency_enc",
    "Payment Currency_enc",
    "From Bank_enc",
    "To Bank_enc",
]

TARGET = "Is Laundering"


def find_best_threshold(y_true, y_prob, beta: float = 1.0) -> float:
    thresholds = np.linspace(0.01, 0.99, 200)
    best_threshold = 0.5
    best_score = 0.0

    for threshold in thresholds:
        pred = (y_prob >= threshold).astype(int)
        precision = precision_score(y_true, pred, zero_division=0)
        recall = recall_score(y_true, pred, zero_division=0)

        if precision + recall == 0:
            continue

        fb = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)
        if fb > best_score:
            best_score = fb
            best_threshold = threshold

    return best_threshold


## 6. Training Function


In [ ]:
def train(df: pd.DataFrame):
    X = df[FEATURE_COLS].copy()
    y = df[TARGET].values

    pos = y.sum()
    neg = len(y) - pos
    scale_pos_weight = neg / pos
    log.info(f"Class balance neg={neg:,} pos={pos:,} scale_pos_weight={scale_pos_weight:.1f}")

    lgb_params = {
        "objective": "binary",
        "metric": ["auc", "average_precision"],
        "boosting_type": "gbdt",
        "num_leaves": 127,
        "max_depth": -1,
        "learning_rate": 0.05,
        "n_estimators": 600,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "scale_pos_weight": scale_pos_weight,
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "verbose": -1,
    }

    mlflow.set_tracking_uri(MLFLOW_URI)
    mlflow.set_registry_uri(MLFLOW_URI)
    mlflow.set_experiment(EXPERIMENT)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    oof_proba = np.zeros(len(y), dtype=np.float32)
    feature_importances = np.zeros(len(FEATURE_COLS))

    with mlflow.start_run(run_name="lgbm_aml"):
        mlflow.log_params(lgb_params)
        mlflow.log_param("n_splits", N_SPLITS)
        mlflow.log_param("train_rows", len(df))
        mlflow.log_param("laundering_rate", float(y.mean()))

        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
            log.info(f"Fold {fold}/{N_SPLITS} ...")

            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model = lgb.LGBMClassifier(**lgb_params)
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
            )

            val_proba = model.predict_proba(X_val)[:, 1]
            oof_proba[val_idx] = val_proba
            feature_importances += model.feature_importances_

            fold_auc = roc_auc_score(y_val, val_proba)
            fold_ap = average_precision_score(y_val, val_proba)
            log.info(f"  Fold {fold}: ROC-AUC={fold_auc:.4f} AP={fold_ap:.4f}")

        best_threshold = find_best_threshold(y, oof_proba)
        y_pred = (oof_proba >= best_threshold).astype(int)

        metrics = {
            "oof_roc_auc": roc_auc_score(y, oof_proba),
            "oof_avg_prec": average_precision_score(y, oof_proba),
            "oof_f1": f1_score(y, y_pred),
            "oof_precision": precision_score(y, y_pred),
            "oof_recall": recall_score(y, y_pred),
            "best_threshold": best_threshold,
        }
        mlflow.log_metrics(metrics)

        log.info("\n" + "=" * 60)
        log.info("OOF Results:")
        for key, value in metrics.items():
            log.info(f"  {key}: {value:.4f}")

        log.info("\nClassification Report (OOF):")
        log.info("\n" + classification_report(y, y_pred, target_names=["Normal", "Laundering"]))

        final_model = lgb.LGBMClassifier(**lgb_params)
        log.info("Training final model on full dataset...")
        final_model.fit(X, y, callbacks=[lgb.log_evaluation(100)])

        model_path = os.path.join(MODEL_DIR, "aml_lgbm.pkl")
        meta_path = os.path.join(MODEL_DIR, "model_meta.pkl")

        joblib.dump(final_model, model_path)
        joblib.dump(
            {
                "feature_cols": FEATURE_COLS,
                "best_threshold": best_threshold,
                "metrics": metrics,
                "feature_importances": dict(zip(FEATURE_COLS, feature_importances / N_SPLITS)),
            },
            meta_path,
        )

        mlflow.lightgbm.log_model(final_model.booster_, "model")
        mlflow.log_artifact(model_path)
        mlflow.log_artifact(meta_path)

        log.info(f"Model saved -> {model_path}")

        log.info("Computing SHAP values on sample...")
        sample_idx = np.random.choice(len(X), min(1000, len(X)), replace=False)
        explainer = shap.TreeExplainer(final_model)
        shap_values = explainer.shap_values(X.iloc[sample_idx])
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

        shap_df = (
            pd.DataFrame(np.abs(shap_values).mean(axis=0)[np.newaxis, :], columns=FEATURE_COLS)
            .T.rename(columns={0: "mean_abs_shap"})
            .sort_values("mean_abs_shap", ascending=False)
        )

        shap_path = os.path.join(MODEL_DIR, "shap_importance.csv")
        shap_df.to_csv(shap_path)
        mlflow.log_artifact(shap_path)

        log.info("\nTop-10 SHAP features:")
        log.info("\n" + shap_df.head(10).to_string())

        run_id = mlflow.active_run().info.run_id
        log.info(f"\nMLflow run_id: {run_id}")
        log.info(f"View UI with: mlflow ui --backend-store-uri {MLFLOW_URI}")

    return final_model, metrics


## 7. Feature Engineering and Training Run


In [ ]:
started_at = time.time()

df_features = engineer_features(df_raw)
model, metrics = train(df_features)

total_minutes = (time.time() - started_at) / 60
log.info(f"Done in {total_minutes:.1f} min")
metrics
